# STE + CW Reparameterization — Vocabulary Collapse Fix

**Problem**: v6/v7 self-routing uses `argmax(C @ h)` where C is the frozen diagonal abstract projection.
On GSM8K, this collapses to a single token. On ScienceQA it works. Why?

**Hypothesis**: The diagonal C isn't expressive enough to spread hidden states across
all abstract token slots on GSM8K. A learned projection W (identity init, trained via STE)
gives the model a way to route hidden states to diverse codes.

**Architecture change**:
```
Current v6:  h → C @ h → argmax → token_id          (no grad through selection)
Proposed:    h → W → Wh → (C @ Wh) → argmax → token_id   (STE grad through W)
                ↑              ↑            ↑
             learned       frozen C     STE: forward=hard, backward=soft
```

**This notebook**:
1. Setup and vocab collapse baseline  
2. Frozen orthogonal codebook C  
3. Learned projection W (identity init)  
4. STE selection function  
5. Forward pass integration via `inputs_embeds`  
6. Visualize C spread and STE gradient flow  
7. Mini training loop: watch W learn, vocab diversify  
8. Compare collapse: v6 argmax vs STE+CW

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from transformers import AutoTokenizer
from sorl.sorl_wrapper import SorlModelWrapper
from data.pt_dataset import get_dataset

device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "Qwen/Qwen3-0.6B"
N_ABS = 32      # abstract vocab size
K = 8           # abstract tokens per sample
D_SHOW = 64     # dims to show in heatmaps

model = SorlModelWrapper.from_pretrained(MODEL_NAME, abstract_vocab_size_list=[N_ABS])
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model = model.to(device).eval()

bv = int(model.vocab_sizes[0].item())   # base vocab size
d  = model.model.config.hidden_size     # d_model
print(f"Model: {MODEL_NAME} | d_model={d} | base_vocab={bv} | n_abs={N_ABS}")
print(f"Device: {device}")

In [ ]:
ds = get_dataset("gsm8k", split="train", tokenizer=tokenizer, max_length=256)
B = 16  # batch size for demo
batch = {k: torch.stack([ds[i][k] for i in range(B)]).to(device)
         for k in ("input_ids", "attention_mask")}
batch["prompt_len"] = torch.tensor([ds[i]["prompt_len"] for i in range(B)], device=device)

ids  = batch["input_ids"]
attn = batch["attention_mask"]
pl   = batch["prompt_len"]
print(f"Batch: {B} samples | ids shape: {ids.shape}")

## 1. Baseline: vocabulary collapse with v6 argmax(C @ h)

Run several forward passes on GSM8K, collect the abstract token IDs chosen, and count unique ones.

In [ ]:
def get_hidden_at_abs_positions(model, ids, attn, pl, K, bv):
    """Run model forward on full sequence, collect hidden states at K positions after prompt end."""
    B = ids.size(0)
    with torch.no_grad():
        out = model(input_ids=ids, attention_mask=attn, output_hidden_states=True)
        h_last = out.hidden_states[-1]  # (B, L, d)
    # Take hidden state at prompt_end - 1 (last prompt token) to predict first abs token
    # For simplicity, gather at pl-1 (last query token position)
    t_idx = (pl - 1).clamp(min=0, max=h_last.size(1) - 1)
    h_at_prompt_end = h_last[torch.arange(B, device=ids.device), t_idx]  # (B, d)
    return h_at_prompt_end, h_last

# Current v6 abstract projection: lm_head rows [bv : bv+N_ABS] are the codebook C
# For v6 with diagonal, the abstract token id = argmax(lm_head(h)[bv:])
def v6_select(h, model, bv, N_ABS):
    """Select abstract tokens using v6 diagonal projection."""
    with torch.no_grad():
        # Use lm_head to get logits over abstract vocab
        logits_abs = model.model.lm_head(h)[:, bv:bv+N_ABS]  # (B, N_ABS)
        token_ids = logits_abs.argmax(dim=-1)  # (B,)
    return token_ids

# Run over N_BATCHES batches
N_BATCHES = 20
v6_counter = Counter()

for b_start in range(0, min(N_BATCHES * B, len(ds)), B):
    b_end = min(b_start + B, len(ds))
    batch_ids  = torch.stack([ds[i]["input_ids"]      for i in range(b_start, b_end)]).to(device)
    batch_attn = torch.stack([ds[i]["attention_mask"] for i in range(b_start, b_end)]).to(device)
    batch_pl   = torch.tensor([ds[i]["prompt_len"]    for i in range(b_start, b_end)], device=device)
    h_prompt, _ = get_hidden_at_abs_positions(model, batch_ids, batch_attn, batch_pl, K, bv)
    tids = v6_select(h_prompt, model, bv, N_ABS)
    v6_counter.update(tids.tolist())

total_v6 = sum(v6_counter.values())
print(f"v6 argmax | {total_v6} selections | {len(v6_counter)} unique tokens (out of {N_ABS})")
print(f"Top-5: {v6_counter.most_common(5)}")
print(f"Entropy: {-sum((c/total_v6)*np.log(c/total_v6+1e-9) for c in v6_counter.values()):.3f} (max={np.log(N_ABS):.3f})")

## 2. Frozen Orthogonal Codebook C

`C ∈ R^{N_ABS × d}` — orthogonal rows, frozen. All abstract codes maximally spread from the start.

In [ ]:
def make_frozen_codebook(N_ABS, d, device):
    """Frozen orthogonal codebook. Rows = abstract token embeddings."""
    C = nn.Embedding(N_ABS, d)
    # Orthogonal init: use QR decomposition of random matrix
    if N_ABS <= d:
        raw = torch.randn(d, N_ABS)
        Q, _ = torch.linalg.qr(raw)    # Q: (d, N_ABS), orthonormal columns
        C.weight.data = Q.T             # (N_ABS, d)
    else:
        # N_ABS > d: fill d orthogonal rows, then random-normalize remaining
        raw = torch.randn(N_ABS, d)
        Q, _ = torch.linalg.qr(raw.T)  # Q: (d, d), use first d rows
        C.weight.data[:d] = Q.T
        rest = torch.randn(N_ABS - d, d)
        C.weight.data[d:] = F.normalize(rest, dim=-1)
    C.weight.requires_grad_(False)      # FROZEN
    return C.to(device)

C = make_frozen_codebook(N_ABS, d, device)

# Verify orthogonality: C @ C^T should be close to identity for N_ABS <= d
gram = C.weight @ C.weight.T   # (N_ABS, N_ABS)
off_diag = gram - torch.eye(N_ABS, device=device)
print(f"Codebook C: shape={C.weight.shape}, requires_grad={C.weight.requires_grad}")
print(f"Row norms: min={C.weight.norm(dim=1).min():.4f}, max={C.weight.norm(dim=1).max():.4f}")
print(f"Off-diagonal max (orthogonality check): {off_diag.abs().max():.6f}  (0 = perfect)")

# Visualize Gram matrix
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ax = axes[0]
im = ax.imshow(gram.cpu().numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_title("Gram matrix C @ C^T (should be ~identity)")
ax.set_xlabel("code j")
ax.set_ylabel("code i")

ax = axes[1]
im2 = ax.imshow(C.weight.cpu().numpy()[:, :D_SHOW], aspect="auto", cmap="RdBu_r")
plt.colorbar(im2, ax=ax)
ax.set_title(f"Codebook C rows (first {D_SHOW} dims)")
ax.set_xlabel(f"Hidden dim (first {D_SHOW})")
ax.set_ylabel("Abstract token ID")

plt.tight_layout()
plt.show()

## 3. Learned Projection W (identity init)

`W ∈ R^{d × d}` — initialized to identity so that at step 0, `CW = C` (no disruption to training).
This is the only new trainable parameter.

In [ ]:
def make_projection_W(d, device):
    """Learned projection W with identity init."""
    W = nn.Linear(d, d, bias=False)
    nn.init.eye_(W.weight)  # W = I at init → CW = C, no disruption
    return W.to(device)

W = make_projection_W(d, device)

n_params = W.weight.numel()
print(f"W: shape={W.weight.shape} | params={n_params:,} ({n_params/1e6:.2f}M)")
print(f"W.weight requires_grad={W.weight.requires_grad}")
print(f"||W - I||_F = {(W.weight - torch.eye(d, device=device)).norm():.6f}  (0 at init)")

# Verify: CW @ C^T should also be ~identity at init (since W=I)
CW = C.weight @ W.weight.T     # (N_ABS, d) — rows of CW
gram_cw = CW @ CW.T
off_cw = gram_cw - torch.eye(N_ABS, device=device)
print(f"||gram(CW) - I||_F at init = {off_cw.norm():.6f}  (should match C at init)")

## 4. STE Selection Function

**Forward**: hard argmax → discrete token ID → look up frozen C row  
**Backward**: soft softmax → weighted sum of C rows → gradient flows through W

In [ ]:
def select_abstract_token_ste(h, W, C, temperature=1.0):
    """
    h: (B, d) — hidden state at abstract token positions
    W: nn.Linear(d, d) — learned projection (trained via STE)
    C: nn.Embedding(N_ABS, d) — frozen orthogonal codebook

    Returns:
        ste_embed: (B, d) — embedding to use as input, STE gradient flows through W
        token_ids: (B,)   — discrete selection (for logging / next-step input)
    """
    Wh = W(h)                                       # (B, d), differentiable through W
    logits = Wh @ C.weight.T / temperature          # (B, N_ABS), differentiable

    # FORWARD: hard argmax
    token_ids = logits.argmax(dim=-1)               # (B,), no grad
    hard_embed = C(token_ids)                        # (B, d), no grad (C frozen)

    # BACKWARD: soft softmax (gradient proxy)
    soft_probs = F.softmax(logits, dim=-1)           # (B, N_ABS), has grad
    soft_embed = soft_probs @ C.weight               # (B, d), has grad via W

    # STE trick: value = hard, gradient = d(soft)/d(W)
    ste_embed = hard_embed + (soft_embed - soft_embed.detach())

    return ste_embed, token_ids, logits


# Quick test: verify STE gradient flows to W but value = hard embedding
h_test = torch.randn(4, d, device=device, requires_grad=False)
W_test = make_projection_W(d, device)

ste_out, tids_test, lgts = select_abstract_token_ste(h_test, W_test, C)
hard_out = C(tids_test)

print(f"ste_embed shape: {ste_out.shape}")
print(f"||ste_embed - hard_embed|| = {(ste_out - hard_out).norm():.8f}  (should be 0 — value=hard)")

# Check gradient flows to W via backward
loss_test = ste_out.sum()
loss_test.backward()
print(f"W.weight.grad is None? {W_test.weight.grad is None}  (False = gradient flows ✓)")
print(f"||W.weight.grad||_F = {W_test.weight.grad.norm():.4f}")
print(f"Token IDs selected: {tids_test.tolist()}")

## 5. Forward pass integration via `inputs_embeds`

Replace abstract token positions in the embedded input with `ste_embed` from the previous hidden state.

In [ ]:
from sorl.sorl_trainer import get_answer_start_index

def build_compressed_seq(ids, attn, pl, n_abs, bv, pad_id, answer_token_id=820):
    B, L = ids.shape
    ans_start = get_answer_start_index(ids, answer_token_id=answer_token_id)
    valid_len = attn.sum(dim=1)
    ans_len = (valid_len - ans_start).clamp(min=0)
    max_comp_len = int((pl + n_abs + ans_len).max().item())
    comp_data = ids.new_full((B, max_comp_len), pad_id)
    comp_attn = attn.new_zeros(B, max_comp_len)
    placeholder = bv
    for b in range(B):
        p, a, al = pl[b].item(), ans_start[b].item(), int(ans_len[b].item())
        comp_data[b, :p] = ids[b, :p]
        comp_attn[b, :p] = 1
        comp_data[b, p:p+n_abs] = placeholder
        comp_attn[b, p:p+n_abs] = 1
        if al > 0:
            comp_data[b, p+n_abs:p+n_abs+al] = ids[b, a:a+al]
            comp_attn[b, p+n_abs:p+n_abs+al] = 1
    ans_pos_s = pl + n_abs
    return comp_data, comp_attn, ans_start, ans_pos_s


def ste_forward_pass(model, ids, attn, pl, W, C, n_abs, bv, n_iter=3, temperature=1.0):
    """
    One recursion pass with STE abstract token selection.
    Returns: logits, token_ids per position, ste_embeds
    """
    B = ids.size(0)
    comp_data, comp_attn, ans_pos_t, ans_pos_s = build_compressed_seq(
        ids, attn, pl, n_abs, bv, tokenizer.pad_token_id)

    # Abstract positions in compressed seq
    abs_positions = [(pl[b].item(), pl[b].item() + n_abs) for b in range(B)]  # (start, end)

    # Get base token embeddings for the whole compressed sequence
    # embed_tokens handles base vocab; abstract slots (= bv) will be replaced
    embed_layer = model.model.model.embed_tokens
    # Clamp abstract placeholder to valid base range for initial embed (will be overwritten)
    safe_comp = comp_data.clamp(max=bv - 1)
    inputs_embeds = embed_layer(safe_comp).clone()  # (B, L_comp, d)

    iter_token_ids = []  # per-iteration abstract token choices

    for it in range(n_iter):
        # Forward with current inputs_embeds
        out = model.model.model(
            inputs_embeds=inputs_embeds,
            attention_mask=comp_attn,
            output_hidden_states=True,
            use_cache=False,
        )
        h = out.last_hidden_state  # (B, L_comp, d)

        # For each abstract position, use hidden state at previous position
        # Abstract positions are [pl[b] : pl[b]+n_abs] for each b
        new_embeds = inputs_embeds.clone()
        all_token_ids = torch.zeros(B, n_abs, dtype=torch.long, device=ids.device)

        for k in range(n_abs):
            # Position of this abstract token (varies per sample due to diff prompt lengths)
            pos_k = torch.tensor([int(pl[b].item()) + k for b in range(B)], device=ids.device)
            prev_pos = (pos_k - 1).clamp(min=0)
            h_prev = h[torch.arange(B, device=ids.device), prev_pos]  # (B, d)

            ste_emb, tok_ids, _ = select_abstract_token_ste(h_prev, W, C, temperature)
            all_token_ids[:, k] = tok_ids

            # Replace embedding at abstract position k for each sample
            for b in range(B):
                new_embeds[b, pos_k[b], :] = ste_emb[b]

        inputs_embeds = new_embeds
        iter_token_ids.append(all_token_ids.cpu())

    return inputs_embeds, iter_token_ids, comp_attn, ans_pos_s


# Run STE forward (no grad for inspection)
B_demo = 8
ids_d  = ids[:B_demo]
attn_d = attn[:B_demo]
pl_d   = pl[:B_demo]

W_demo = make_projection_W(d, device)

with torch.no_grad():
    _, iter_tids, comp_attn_d, ans_pos_s_d = ste_forward_pass(
        model, ids_d, attn_d, pl_d, W_demo, C, K, bv, n_iter=3)

print(f"STE forward: {len(iter_tids)} iterations, {K} abstract tokens per sample")
for it, tids in enumerate(iter_tids):
    unique_per_sample = [len(set(tids[b].tolist())) for b in range(B_demo)]
    all_unique = len(set(tids.flatten().tolist()))
    print(f"  Iter {it}: unique tokens total={all_unique}/{N_ABS} | per-sample avg={np.mean(unique_per_sample):.1f}")

## 6. Visualize: Codebook spread + STE token distributions

In [ ]:
# Collect STE token frequencies over many batches (untrained W = identity = same as using C directly)
ste_counter_before = Counter()
N_BATCHES_VIZ = 20

W_fresh = make_projection_W(d, device)  # untrained (identity)

for b_start in range(0, min(N_BATCHES_VIZ * 4, len(ds)), 4):
    b_end = min(b_start + 4, len(ds))
    bids  = torch.stack([ds[i]["input_ids"]      for i in range(b_start, b_end)]).to(device)
    battn = torch.stack([ds[i]["attention_mask"] for i in range(b_start, b_end)]).to(device)
    bpl   = torch.tensor([ds[i]["prompt_len"]    for i in range(b_start, b_end)], device=device)
    with torch.no_grad():
        _, tids_list, _, _ = ste_forward_pass(model, bids, battn, bpl, W_fresh, C, K, bv, n_iter=1)
    ste_counter_before.update(tids_list[0].flatten().tolist())

# Plot comparison: v6 vs STE (untrained W)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# (a) v6 vocab distribution
ax = axes[0]
v6_counts = [v6_counter.get(i, 0) for i in range(N_ABS)]
ax.bar(range(N_ABS), v6_counts, color="#c44e52")
ax.set_title(f"v6 argmax | {len(v6_counter)}/{N_ABS} tokens used")
ax.set_xlabel("Abstract token ID")
ax.set_ylabel("Count")

# (b) STE (W=I) vocab distribution
ax = axes[1]
ste_b_counts = [ste_counter_before.get(i, 0) for i in range(N_ABS)]
ax.bar(range(N_ABS), ste_b_counts, color="#4c72b0")
ax.set_title(f"STE (W=I, untrained) | {len(ste_counter_before)}/{N_ABS} tokens used")
ax.set_xlabel("Abstract token ID")
ax.set_ylabel("Count")

# (c) Codebook cosine similarity matrix
ax = axes[2]
Cn = F.normalize(C.weight, dim=1)  # normalized
cos_mat = (Cn @ Cn.T).cpu().detach().numpy()
im = ax.imshow(cos_mat, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_title("C cosine similarity (orthogonal → diagonal)")
ax.set_xlabel("Token j")
ax.set_ylabel("Token i")

plt.tight_layout()
plt.show()

v6_ent   = -sum((c/sum(v6_counts))*np.log(c/sum(v6_counts)+1e-9) for c in v6_counts if c > 0)
ste_b_ent = -sum((c/sum(ste_b_counts))*np.log(c/sum(ste_b_counts)+1e-9) for c in ste_b_counts if c > 0)
print(f"v6 entropy:          {v6_ent:.3f}  (max={np.log(N_ABS):.3f})")
print(f"STE (W=I) entropy:   {ste_b_ent:.3f}")

## 7. Mini Training Loop — watch W learn and vocab diversify

Train W for a few steps on `traj_loss` (CE on answer tokens conditioned on abstract prefix).
No other model parameters change — this isolates the effect of the W projection.

In [ ]:
def compute_traj_loss(model, inputs_embeds, comp_attn, ids, pl, bv, n_abs):
    """CE loss on answer (NL) tokens of compressed sequence, given inputs_embeds."""
    out = model.model.model(
        inputs_embeds=inputs_embeds,
        attention_mask=comp_attn,
        use_cache=False,
    )
    logits_full = model.model.lm_head(out.last_hidden_state)  # (B, L_comp, V)
    B, L_comp, V = logits_full.shape

    # Target: shift-by-1, only answer (NL) positions
    # Answer starts at pl + n_abs, mask abstract and query
    lab = torch.full((B, L_comp), -100, dtype=torch.long, device=inputs_embeds.device)
    for b in range(B):
        ans_start = int(pl[b].item()) + n_abs
        valid = int(comp_attn[b].sum().item())
        # Labels: shifted by 1, only NL answer tokens
        if valid > ans_start + 1:
            # Get original answer token ids from the compressed data (NL tokens only)
            # We don't have comp_data here directly, so use comp_attn length
            pass  # handled by caller

    return F.cross_entropy(
        logits_full[:, :-1].contiguous().view(-1, V),
        lab[:, 1:].contiguous().view(-1),
        ignore_index=-100,
    )


# Cleaner training loop with proper labels
W_trained = make_projection_W(d, device)
optim_W = torch.optim.Adam(W_trained.parameters(), lr=1e-3)

N_TRAIN_STEPS = 60
TRAIN_B = 4
N_ITER_RECURSE = 2

losses_train, entropy_train = [], []
W_drift = []  # ||W - I||_F over steps

model.train()
for step in range(N_TRAIN_STEPS):
    b_start = (step * TRAIN_B) % (len(ds) - TRAIN_B)
    bids  = torch.stack([ds[i]["input_ids"]      for i in range(b_start, b_start+TRAIN_B)]).to(device)
    battn = torch.stack([ds[i]["attention_mask"] for i in range(b_start, b_start+TRAIN_B)]).to(device)
    bpl   = torch.tensor([ds[i]["prompt_len"]    for i in range(b_start, b_start+TRAIN_B)], device=device)

    comp_data_s, comp_attn_s, _, ans_pos_s_s = build_compressed_seq(
        bids, battn, bpl, K, bv, tokenizer.pad_token_id)

    # Build inputs_embeds for compressed seq, replace abstract slots via STE
    embed_layer = model.model.model.embed_tokens
    safe_comp = comp_data_s.clamp(max=bv-1)
    inputs_embeds_s = embed_layer(safe_comp).clone().detach()  # start detached

    # STE forward N_ITER_RECURSE times
    for it in range(N_ITER_RECURSE):
        with torch.no_grad():
            out_h = model.model.model(
                inputs_embeds=inputs_embeds_s,
                attention_mask=comp_attn_s,
                output_hidden_states=True,
                use_cache=False,
            )
            h_all = out_h.last_hidden_state  # (B, L, d)

        new_embeds = inputs_embeds_s.clone().detach()
        for k in range(K):
            pos_k = bpl + k  # (B,)
            prev_pos = (pos_k - 1).clamp(min=0)
            h_prev = h_all[torch.arange(TRAIN_B, device=device), prev_pos]  # (B, d)
            ste_emb, _, _ = select_abstract_token_ste(h_prev, W_trained, C)
            for b in range(TRAIN_B):
                new_embeds[b, pos_k[b], :] = ste_emb[b]
        inputs_embeds_s = new_embeds

    # Final forward: compute traj_loss on answer tokens
    out_final = model.model.model(
        inputs_embeds=inputs_embeds_s,
        attention_mask=comp_attn_s,
        use_cache=False,
    )
    logits_final = model.model.lm_head(out_final.last_hidden_state)  # (B, L, V)

    # Labels: NL answer tokens only
    L_comp = comp_data_s.size(1)
    lab = comp_data_s.clone().long()
    for b in range(TRAIN_B):
        ans_start_b = int(bpl[b].item()) + K
        lab[b, :ans_start_b] = -100   # mask query + abstract
        lab[b, lab[b] >= bv] = -100   # mask any remaining abstract tokens
        lab[b, comp_attn_s[b] == 0] = -100  # mask padding

    traj_loss = F.cross_entropy(
        logits_final[:, :-1].reshape(-1, logits_final.size(-1)),
        lab[:, 1:].reshape(-1),
        ignore_index=-100,
    )

    optim_W.zero_grad()
    traj_loss.backward()
    optim_W.step()

    losses_train.append(traj_loss.item())
    W_drift.append((W_trained.weight - torch.eye(d, device=device)).norm().item())

    # Measure entropy of token selection this step
    with torch.no_grad():
        h_step = h_all[torch.arange(TRAIN_B, device=device), (bpl - 1).clamp(min=0)]
        _, tids_step, lgts_step = select_abstract_token_ste(h_step, W_trained, C)
        probs_step = F.softmax(lgts_step, dim=-1).cpu().numpy()
        ent_step = float(-np.sum(probs_step * np.log(probs_step + 1e-9), axis=-1).mean())
    entropy_train.append(ent_step)

    if (step + 1) % 10 == 0:
        print(f"Step {step+1:3d} | traj_loss={traj_loss.item():.4f} | "
              f"entropy={ent_step:.3f} | ||W-I||={W_drift[-1]:.4f}")

model.eval()

## 8. Compare: Before vs After W training — vocab diversity

In [ ]:
# Collect token frequencies after W training
ste_counter_after = Counter()
for b_start in range(0, min(N_BATCHES_VIZ * 4, len(ds)), 4):
    b_end = min(b_start + 4, len(ds))
    bids  = torch.stack([ds[i]["input_ids"]      for i in range(b_start, b_end)]).to(device)
    battn = torch.stack([ds[i]["attention_mask"] for i in range(b_start, b_end)]).to(device)
    bpl   = torch.tensor([ds[i]["prompt_len"]    for i in range(b_start, b_end)], device=device)
    with torch.no_grad():
        _, tids_list, _, _ = ste_forward_pass(model, bids, battn, bpl, W_trained, C, K, bv, n_iter=1)
    ste_counter_after.update(tids_list[0].flatten().tolist())

# --- Plots ---
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Row 1: Token frequency histograms
for ax, counter, title, color in zip(
    axes[0],
    [v6_counter, ste_counter_before, ste_counter_after],
    ["v6 (argmax C@h)", "STE W=I (untrained)", f"STE trained ({N_TRAIN_STEPS} steps)"],
    ["#c44e52", "#4c72b0", "#55a868"],
):
    counts = [counter.get(i, 0) for i in range(N_ABS)]
    total  = sum(counts)
    n_used = sum(1 for c in counts if c > 0)
    ent    = -sum((c/total)*np.log(c/total+1e-9) for c in counts if c > 0)
    ax.bar(range(N_ABS), counts, color=color)
    ax.set_title(f"{title}\n{n_used}/{N_ABS} tokens used | entropy={ent:.2f}")
    ax.set_xlabel("Abstract token ID")
    ax.set_ylabel("Count")

# Row 2: Training curves
ax = axes[1][0]
ax.plot(losses_train, color="#c44e52")
ax.set_title("Traj loss during W training")
ax.set_xlabel("Step")
ax.set_ylabel("CE loss")
ax.grid(True, alpha=0.3)

ax = axes[1][1]
ax.plot(entropy_train, color="#4c72b0")
ax.axhline(np.log(N_ABS), color="black", ls="--", lw=0.8, label=f"max={np.log(N_ABS):.2f}")
ax.set_title("Token selection entropy during W training")
ax.set_xlabel("Step")
ax.set_ylabel("Entropy (nats)")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1][2]
ax.plot(W_drift, color="#dd8452")
ax.set_title("||W - I||_F (drift from identity init)")
ax.set_xlabel("Step")
ax.set_ylabel("Frobenius norm")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Summary
print("\n=== Summary ===")
for name, counter in [("v6", v6_counter), ("STE W=I", ste_counter_before), ("STE trained", ste_counter_after)]:
    counts = [counter.get(i, 0) for i in range(N_ABS)]
    total  = sum(counts)
    n_used = sum(1 for c in counts if c > 0)
    ent    = -sum((c/total)*np.log(c/total+1e-9) for c in counts if c > 0)
    print(f"  {name:20s}: {n_used:2d}/{N_ABS} tokens | entropy={ent:.3f}/{np.log(N_ABS):.3f}")

## 9. CW vs C: how much does W rotate the codebook?

Visualize `(CW)` rows vs `C` rows — do they stay spread after training?

In [ ]:
with torch.no_grad():
    CW_rows = C.weight @ W_trained.weight.T   # (N_ABS, d) — effective codebook
    CW_norm = F.normalize(CW_rows, dim=1)
    C_norm  = F.normalize(C.weight, dim=1)

    cos_C  = (C_norm  @ C_norm.T).cpu().numpy()
    cos_CW = (CW_norm @ CW_norm.T).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
im = ax.imshow(cos_C, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_title("Cosine sim: C (frozen)")

ax = axes[1]
im = ax.imshow(cos_CW, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_title("Cosine sim: CW (after training)")

ax = axes[2]
diff = cos_CW - cos_C
im = ax.imshow(diff, cmap="RdBu_r", vmin=-0.2, vmax=0.2)
plt.colorbar(im, ax=ax)
ax.set_title("Δ cosine sim (CW - C)")

plt.tight_layout()
plt.show()

# Off-diagonal statistics
mask = ~np.eye(N_ABS, dtype=bool)
print(f"C  off-diag cosine sim: mean={cos_C[mask].mean():.4f}, max={cos_C[mask].max():.4f}")
print(f"CW off-diag cosine sim: mean={cos_CW[mask].mean():.4f}, max={cos_CW[mask].max():.4f}")
print(f"||W_trained - I||_F = {(W_trained.weight - torch.eye(d, device=device)).norm().item():.4f}")